<a href="https://colab.research.google.com/github/RaiSandip25/Bert_Model/blob/main/bert_sentiment_sst2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis with BERT — SST-2 Dataset
**Task:** Classify sentences as `Positive` or `Negative` sentiment

**Labels:**
- `0` → Negative
- `1` → Positive

**Model:** `bert-base-uncased` → `BertForSequenceClassification`  
**Dataset:** SST-2 (Stanford Sentiment Treebank) — ~67K short sentences  
**Training time:** ~10–15 mins on GPU (much faster than IMDb due to short sentences)

## 1. Install & Import

In [ ]:
!pip install transformers[torch] datasets scikit-learn accelerate -q

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Load Dataset
SST-2 loads directly from HuggingFace — no manual download needed.

In [ ]:
dataset = load_dataset('glue', 'sst2')
print(dataset)
print("\nSample examples:")
for i in range(3):
    ex = dataset['train'][i]
    label = 'Positive' if ex['label'] == 1 else 'Negative'
    print(f"  [{label}] {ex['sentence']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Sample examples:
  [Negative] hide new secretions from the parental units 
  [Negative] contains no wit , only labored gags 
  [Positive] that loves its characters and communicates something rather beautiful about human nature 


## 3. Tokenization
BERT expects tokenized input with `input_ids`, `attention_mask`, and `token_type_ids`.
We use `truncation=True` and `max_length=128` (SST-2 sentences are short — this keeps things fast).

In [ ]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['sentence'],
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

# Rename label column so Trainer recognizes it
tokenized_dataset = tokenized_dataset.rename_column('label', 'labels')
tokenized_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels']
)

train_dataset = tokenized_dataset['train']
val_dataset   = tokenized_dataset['validation']

print(f"Train size : {len(train_dataset)}")
print(f"Val size   : {len(val_dataset)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Train size : 67349
Val size   : 872


## 4. Load Pre-trained BERT Model
`BertForSequenceClassification` adds a linear classifier on top of BERT's `[CLS]` token output.
We set `num_labels=2` for binary classification.

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'Negative', 1: 'Positive'},
    label2id={'Negative': 0, 'Positive': 1},
)
model.to(device)
print(f"Model loaded — parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded — parameters: 109,483,778


## 5. Training Setup

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='binary')
    return {'accuracy': acc, 'f1': f1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./bert-sst2-output',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # Mixed precision on GPU
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 6. Train the Model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.152295,0.225255,0.930046,0.931073
2,0.109408,0.262611,0.915138,0.919037
3,0.069996,0.293108,0.925459,0.927697


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6315, training_loss=0.1464463847078035, metrics={'train_runtime': 609.5058, 'train_samples_per_second': 331.493, 'train_steps_per_second': 10.361, 'total_flos': 4176656240000220.0, 'train_loss': 0.1464463847078035, 'epoch': 3.0})

## 7. Evaluate on Validation Set

In [ ]:
results = trainer.evaluate()
print(f"\nValidation Accuracy : {results['eval_accuracy']:.4f}")
print(f"Validation F1       : {results['eval_f1']:.4f}")


Validation Accuracy : 0.9335
Validation F1       : 0.9344


In [ ]:
# Full classification report
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

              precision    recall  f1-score   support

    Negative       0.93      0.94      0.93       428
    Positive       0.94      0.93      0.93       444

    accuracy                           0.93       872
   macro avg       0.93      0.93      0.93       872
weighted avg       0.93      0.93      0.93       872



## 8. Inference — Predict on Custom Sentences

In [ ]:
def predict_sentiment(texts):
    """Predict sentiment for a list of sentences."""
    model.eval()
    inputs = tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

    print("-" * 65)
    for text, pred, prob in zip(texts, preds, probs):
        label = 'Positive 😊' if pred == 1 else 'Negative 😞'
        confidence = prob[pred] * 100
        print(f"Text       : {text}")
        print(f"Prediction : {label}  (confidence: {confidence:.1f}%)")
        print("-" * 65)


# Try it out!
predict_sentiment([
    "This movie was absolutely wonderful and heartwarming.",
    "A complete waste of time. Terrible acting and boring plot.",
    "The film had some great moments but the ending was disappointing.",
    "One of the best performances I have ever seen!",
])

-----------------------------------------------------------------
Text       : This movie was absolutely wonderful and heartwarming.
Prediction : Positive 😊  (confidence: 99.9%)
-----------------------------------------------------------------
Text       : A complete waste of time. Terrible acting and boring plot.
Prediction : Negative 😞  (confidence: 99.8%)
-----------------------------------------------------------------
Text       : The film had some great moments but the ending was disappointing.
Prediction : Negative 😞  (confidence: 99.6%)
-----------------------------------------------------------------
Text       : One of the best performances I have ever seen!
Prediction : Positive 😊  (confidence: 99.9%)
-----------------------------------------------------------------
